# L3 V1/V2 VADER validation
Run in Google Colab. Branch-only validation; do not merge until evidence reviewed. Labels/confidence are never loaded.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip install vaderSentiment==3.3.2 pytest


In [ ]:
!git clone --branch feature/l3-vader-engineering https://github.com/frankie-chew-nus/sentiment-clustering-ml.git /content/sentiment-clustering-ml
%cd /content/sentiment-clustering-ml
!git branch --show-current
!python -m pytest -q tests/test_feature_engineering.py


In [ ]:
from pathlib import Path
from feature_engineering import load_raw_text_only, extract_vader_frame, raw_csv_fingerprint
from feature_engineering_config import select_vader_columns
RAW_PATH = Path('/content/drive/MyDrive/MSI5001 project/EDA/Tweets-redacted.csv')
assert RAW_PATH.is_file(), RAW_PATH
fingerprint = raw_csv_fingerprint(RAW_PATH)
texts = load_raw_text_only(RAW_PATH)
master = extract_vader_frame(texts)
print('Fingerprint:', fingerprint)
print('Rows:', len(master), 'Columns:', list(master.columns))
print('V1:', select_vader_columns('V1'), 'V2:', select_vader_columns('V2'))
display(master.describe())
print('Compound zeros:', int((master.V_compound == 0).sum()))
print('Both pos/neg zero:', int(((master.V_pos == 0) & (master.V_neg == 0)).sum()))
print('Distinct compound:', master.V_compound.nunique())
assert len(master) == len(texts) == 14639
assert fingerprint['source_sha256'] == 'a41aeefcff9740c090153f608691d345c84f59084e40f03169c2f667df66578e'
assert int((master.V_compound == 0).sum()) == 2323
assert int(((master.V_pos == 0) & (master.V_neg == 0)).sum()) == 2269
assert master.V_compound.nunique() == 2915
print('V0 audit parity checks passed')


## Evidence and review
Record test output, SHA256, score parity, any anomalies, and decision before merging. Export is optional and should be kept in Drive, not committed as a large dataset.

In [ ]:
OUT = Path('/content/drive/MyDrive/MSI5001 project/EDA/Tweets-L3-vader-features.csv')
master.to_csv(OUT, index=False)
print('Saved:', OUT)
